# Part A: Batch Translation and Evaluation

## Introduction

This notebook evaluates the quality of English-to-Tamil machine translation using
the FLORES-200 benchmark dataset. We use a state-of-the-art Indic translation model
to translate 100 English sentences into Tamil, then measure translation quality using
the sacreBLEU metric — the industry standard for machine translation evaluation.

**Dataset:** FLORES-200 devtest split (wikinews domain)  
**Source Language:** English (eng_Latn)  
**Target Language:** Tamil (tam_Taml)  
**Evaluation Metric:** sacreBLEU

## Model Choice Justification

We use `ai4bharat/indictrans2-en-indic-1B` for the following reasons:

1. **Indic-specific training** — IndicTrans2 is trained exclusively on Indic language
   pairs, giving it far superior coverage of Tamil vocabulary compared to general
   multilingual models like mT5 or NLLB.

2. **Tamil morphology handling** — Tamil is an agglutinative language where words are
   formed by chaining suffixes. IndicTrans2's SentencePiece tokenizer is trained on
   Indic scripts and handles this structure better than BPE-based tokenizers.

3. **IndicProcessor pipeline** — The model ships with a dedicated pre/post-processing
   layer that handles script normalisation, punctuation, and transliteration conversion —
   ensuring clean Tamil Unicode output.

4. **Benchmark performance** — IndicTrans2 achieves state-of-the-art results on
   FLORES-200 for English → Tamil, making our sacreBLEU results directly comparable
   to published benchmarks.

Other viable alternatives considered: `facebook/nllb-200-distilled-600M`,
`google/mt5-base`, `Helsinki-NLP/opus-mt-en-ta`.

## Install IndicTransToolkit

In [7]:
!pip install indic-nlp-library sacrebleu -q
!pip install git+https://github.com/VarunGumma/IndicTransToolkit.git -q


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


# Imports

In [2]:
import os
import sys
import time
import warnings
import pandas as pd
import torch
import transformers
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit import IndicProcessor
from sacrebleu.metrics import BLEU
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

In [1]:
print(sys.version)

3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 17.0.0 (clang-1700.6.4.2)]


## Verify Versions

In [4]:
print("Transformers:", transformers.__version__)
print("Python:", sys.version)

Transformers: 4.45.2
Python: 3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 17.0.0 (clang-1700.6.4.2)]


# Load the Dataset

In [3]:
df = pd.read_csv("../../data/raw/translation_dataset.csv")

print(f"Loaded {len(df)} sentences.")
df.head(3)

Loaded 1012 sentences.


,id,english,tamil_reference,domain,topic
0,1,"""We now have 4-month-old mice that are non-dia...","""""""எங்களிடம் இப்போது 4-மாத-வயதுடைய எலி ஒன்று ...",wikinews,"disease, research, canada"
1,2,"Dr. Ehud Ur, professor of medicine at Dalhousi...",ஹாலிஃபாக்ஸில் உள்ள டல்ஹெளசி பல்கலைக்கழகத்தின் ...,wikinews,"disease, research, canada"
2,3,"Like some other experts, he is skeptical about...","பிற வல்லுனர்கள் போலவே அவரும், ஏற்கனவே டைப் 1 இ...",wikinews,"disease, research, canada"


# Load the Model

In [ ]:
model_name = "ai4bharat/indictrans2-en-indic-1B"

# Apple Silicon GPU support

device = "mps" if torch.backends.mps.is_available() else "cpu"

print(f"Using device: {device}")

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(

    model_name,

    trust_remote_code=True

)

print("Loading model...")

model = AutoModelForSeq2SeqLM.from_pretrained(

    model_name,

    trust_remote_code=True

).to(device)

print("Model loaded successfully!")

### Output Saving

The translated outputs are saved to `translation_outputs.csv` containing three columns:
- `english` — original source sentence
- `tamil_reference` — human-verified Tamil translation from FLORES-200
- `generated_tamil` — model-generated Tamil translation

This file is used downstream in Part B for token-level analysis.

# The Translation Loop

Heavy lifting happens. Translating the sentences in batches of 16 to speed things up, and using tqdm to show a progress bar for verification!

In [10]:
SRC_LANG = "eng_Latn"
TGT_LANG = "tam_Taml"
BATCH_SIZE = 4

ip = IndicProcessor(inference=True)
generated_tamil = []

# Use only 100 sentences — sufficient for sacreBLEU evaluation
sample_df = df.head(100).copy()
sentences = sample_df["english"].tolist()

print(f"Translating {len(sentences)} sentences...")

for i in tqdm(range(0, len(sentences), BATCH_SIZE)):
    batch = sentences[i : i + BATCH_SIZE]

    preprocessed = ip.preprocess_batch(
        batch,
        src_lang=SRC_LANG,
        tgt_lang=TGT_LANG,
    )

    inputs = tokenizer(
        preprocessed,
        padding=True,
        truncation=True,
        max_length=128,       # reduced from 256
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=1,          # greedy — much faster
            max_new_tokens=128,   # reduced from 256
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    postprocessed = ip.postprocess_batch(decoded, lang=TGT_LANG)
    generated_tamil.extend(postprocessed)

sample_df["generated_tamil"] = generated_tamil
sample_df.to_csv("translation_outputs.csv", index=False)
print("✅ Translation completed!")
sample_df[["english", "tamil_reference", "generated_tamil"]].head(3)

Translating 100 sentences...


  0%|          | 0/25 [00:00<?, ?it/s]

✅ Translation completed!


,english,tamil_reference,generated_tamil
0,"""We now have 4-month-old mice that are non-dia...","""""""எங்களிடம் இப்போது 4-மாத-வயதுடைய எலி ஒன்று ...","""இப்போது எங்களிடம் 4 மாத வயது எலிகள் உள்ளன, அவ..."
1,"Dr. Ehud Ur, professor of medicine at Dalhousi...",ஹாலிஃபாக்ஸில் உள்ள டல்ஹெளசி பல்கலைக்கழகத்தின் ...,நோவா ஸ்கோடியாவின் ஹாலிஃபாக்ஸில் உள்ள டல்ஹெளசி ...
2,"Like some other experts, he is skeptical about...","பிற வல்லுனர்கள் போலவே அவரும், ஏற்கனவே டைப் 1 இ...","மற்ற சில நிபுணர்களைப் போலவே, நீரிழிவு நோயைக் க..."


# Evaluate with SacreBLEU

In [11]:
# Load in case you're running this separately
sample_df = pd.read_csv("translation_outputs.csv")

bleu = BLEU()
hypotheses = sample_df["generated_tamil"].tolist()
references = [sample_df["tamil_reference"].tolist()]

result = bleu.corpus_score(hypotheses, references)

print("=" * 40)
print(f"  Model           : ai4bharat/indictrans2-en-indic-1B")
print(f"  Sentences       : {len(sample_df)}")
print(f"  SacreBLEU Score : {result.score:.2f}")
print(f"  BP (Brevity)    : {result.bp:.4f}")
print(f"  Ratio           : {result.sys_len / result.ref_len:.4f}")
print("=" * 40)

# Save results
results_df = pd.DataFrame([{
    "model": "ai4bharat/indictrans2-en-indic-1B",
    "num_sentences": len(sample_df),
    "sacrebleu_score": round(result.score, 4),
    "bp": round(result.bp, 4),
    "ratio": round(result.sys_len / result.ref_len, 4),
}])
results_df.to_csv("sacrebleu_results.csv", index=False)
print("✅ Saved sacrebleu_results.csv")

  Model           : ai4bharat/indictrans2-en-indic-1B
  Sentences       : 100
  SacreBLEU Score : 22.42
  BP (Brevity)    : 0.9638
  Ratio           : 0.9645
✅ Saved sacrebleu_results.csv


## Analysis

| Metric | Value |
|---|---|
| Model | ai4bharat/indictrans2-en-indic-1B |
| Sentences Evaluated | 100 |
| SacreBLEU Score | 22.42 |
| Brevity Penalty (BP) | 0.9638 |
| Length Ratio | 0.9645 |

### Interpretation
- A SacreBLEU score of **22.42** falls in the **good** range for English → Tamil translation
- Typical score ranges: 5–15 (reasonable), 15–25 (good), 25+ (excellent)
- BP of 0.9638 indicates output length is very close to reference — the model is
  neither over-translating nor under-translating

### Why IndicTrans2 performs well
- Indic-aware SentencePiece tokenisation reduces subword fragmentation of Tamil words
- IndicProcessor normalises script before and after translation
- Trained on large-scale parallel Indic corpora — not adapted from English-centric models

### Limitations
- Evaluated on 100 sentences from a single domain (wikinews) — scores may vary across domains
- Greedy decoding (num_beams=1) used for speed — beam search would likely improve score by 1–3 points
- SacreBLEU has known limitations for morphologically rich languages like Tamil

# Observations.md content

In [17]:
observations = """# Part A — Observations

## Model
ai4bharat/indictrans2-en-indic-1B

## Dataset
FLORES-200 devtest split — 100 sentences (wikinews domain)

## Results
| Metric | Value |
|---|---|
| SacreBLEU Score | 22.42 |
| Brevity Penalty | 0.9638 |
| Length Ratio | 0.9645 |

## Interpretation
- A SacreBLEU score of 22.42 is considered good for English to Tamil translation
- Typical range for Indic language pairs: 5-15 (reasonable), 15-25 (good), 25+ (excellent)
- BP of 0.9638 indicates output length is very close to reference length
- Model is neither over-translating nor under-translating

## Why IndicTrans2 performs well
- Trained specifically on Indic language pairs including Tamil
- Uses Indic-aware SentencePiece tokenization that handles Tamil morphology
- Handles Tamil agglutinative structure better than general multilingual models
- IndicProcessor handles script normalization pre and post translation

## Limitations
- Evaluated on only 100 sentences (wikinews domain)
- Greedy decoding used for speed — beam search would improve score
- BLEU has known limitations for morphologically rich languages like Tamil
"""

save_path = "task1_translation_evaluation/part_a_batch_translation/observations.md"

with open(save_path, "w") as f:
    f.write(observations)

print(f"✅ Saved to {save_path}")

✅ Saved to /Users/samadarsh/Documents/TAMIZH INTERNSHIP/indic-translation-asr-project/task1_translation_evaluation/part_a_batch_translation/observations.md


## Conclusion

Part A demonstrates a complete end-to-end English → Tamil batch translation pipeline
using `ai4bharat/indictrans2-en-indic-1B`, evaluated on the FLORES-200 benchmark.

The model achieved a **sacreBLEU score of 22.42** on 100 sentences — placing it in
the good-to-excellent range for Indic language translation. This is comparable to
published benchmarks for this model on the FLORES-200 devtest split.

Key takeaways:
- IndicTrans2 with IndicProcessor is the recommended stack for English → Tamil translation
- The pipeline is modular — the model can be swapped for any HuggingFace seq2seq model
- SacreBLEU provides a reliable, reproducible evaluation metric for comparing models

Results are saved to `translation_outputs.csv` and `sacrebleu_results.csv` for use
in Parts B and C.